In [ ]:
import os
import torch
import numpy as np
from torch import nn, optim
# Import necessary schedulers
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
# Import WeightedRandomSampler
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import timm # Use timm to create EfficientNet
from PIL import Image
import logging
import random
import matplotlib.pyplot as plt
import seaborn as sns
from torch.amp import GradScaler, autocast
from sklearn.model_selection import train_test_split
import glob  # For listing files
# Removed unused Transformer/Attention imports
# Mixed precision training tools
from torch.cuda.amp import GradScaler, autocast # Use torch.amp
import torch
print(f"PyTorch Version: {torch.__version__}")
# Also check CUDA availability just in case
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
# Configuration class (Adapted for Single Model Training)
class Config:
    IMG_SIZE = 224
    BATCH_SIZE = 32
    NUM_CLASSES = 2 # CASIA: Authentic vs Tampered
    EPOCHS = 15 # Total epochs for fine-tuning (Adjust as needed)
    WARMUP_EPOCHS = 3 # Warmup epochs
    LEARNING_RATE = 1e-4 # Learning rate for fine-tuning
    WEIGHT_DECAY = 0.01
    PREPROCESSED_DIR = "/content/drive/MyDrive/preprocessed_casia"
    DATASET_FRACTION = 1.0 # Use 1.0 for full dataset, smaller value for testing
    EARLY_STOPPING_PATIENCE = 5 # Patience pour l'arrêt anticipé
    CLIP_GRAD_NORM = 1.0
    SEED = 42
    TRAIN_SIZE = 0.7
    VAL_SIZE = 0.15
    TEST_SIZE = 0.15
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    DATA_PATH = "/content/drive/MyDrive/CASIA2.0_revised_corrected/casia" # Make sure this path is correct
    MODEL_NAME = "efficientnet_b0_casia_bal" # Model name reflects EfficientNet-B0
from google.colab import drive
drive.mount('/content/drive')
logging.info("Google Drive mounted.")
config = Config()

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.SEED)

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Using device: {DEVICE}")
DEVICE_TYPE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Mount Google Drive (Uncomment if using Colab and data is on Drive) ---
# from google.colab import drive
# drive.mount('/content/drive')
# logging.info("Google Drive mounted.")


# --- Dataset Definition ---
class CASIADataset(Dataset):
    def __init__(self, images, labels, transforms):
        self.images = images; self.labels = labels; self.transforms = transforms
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        image = self.images[idx]; label = self.labels[idx]
        try:
            # Ensure image is uint8 before converting to PIL Image
            if image.dtype != np.uint8:
                # Check if data is normalized (0-1 range) or 0-255 range
                if image.max() <= 1.0 and image.min() >= 0.0:
                    image = (image * 255).astype(np.uint8)
                else:
                    image = image.astype(np.uint8) # Assume it's already 0-255

            image = Image.fromarray(image).convert('RGB') # Ensure RGB
            image = self.transforms(image)
        except Exception as e: logging.error(f"Error processing image at index {idx}: {e}"); raise e
        return image, torch.tensor(label, dtype=torch.long)

# --- Data Loading/Preprocessing (Mostly unchanged) ---
def load_or_preprocess_data(data_path, preprocessed_dir, img_size, dataset_fraction=1.0, train_split=0.8, val_split=0.1, test_split=0.1, seed=42):
    if abs(train_split + val_split + test_split - 1.0) > 1e-6: raise ValueError("Splits must sum to 1.0")
    train_file=os.path.join(preprocessed_dir, f"train_data_{img_size}.npz"); val_file=os.path.join(preprocessed_dir, f"val_data_{img_size}.npz"); test_file=os.path.join(preprocessed_dir,f"test_data_{img_size}.npz")
    try:
        if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file):
            logging.info(f"Loading preprocessed data ({img_size}x{img_size}) from {preprocessed_dir}...");
            with np.load(train_file) as d: train_images, train_labels = d['images'], d['labels']
            with np.load(val_file) as d: val_images, val_labels = d['images'], d['labels']
            with np.load(test_file) as d: test_images, test_labels = d['images'], d['labels']
            logging.info("Loaded preprocessed data.")
        else:
            logging.info(f"Processing dataset from {data_path} for size {img_size}x{img_size}...")
            auth_dir, tamp_dir = os.path.join(data_path,"Au"), os.path.join(data_path,"Tp")
            if not os.path.isdir(auth_dir) or not os.path.isdir(tamp_dir): raise FileNotFoundError(f"Dataset directories not found at {data_path}")
            exts = ('*.jpg','*.jpeg','*.png','*.tif','*.tiff','*.bmp'); auth_p,tamp_p=[],[]
            for ext in exts:
                auth_p.extend(glob.glob(os.path.join(auth_dir,ext))); auth_p.extend(glob.glob(os.path.join(auth_dir,ext.upper())))
                tamp_p.extend(glob.glob(os.path.join(tamp_dir,ext))); tamp_p.extend(glob.glob(os.path.join(tamp_dir,ext.upper())))
            auth_p,tamp_p=sorted(list(set(auth_p))),sorted(list(set(tamp_p)))
            logging.info(f"Found {len(auth_p)} Authentic, {len(tamp_p)} Tampered image paths.")
            if not auth_p and not tamp_p: raise FileNotFoundError("No image files found in Au/Tp directories")

            images, labels, proc_cnt, total_f = [], [], 0, len(auth_p)+len(tamp_p)
            target_f = int(total_f * dataset_fraction); logging.info(f"Processing up to {target_f} images ({dataset_fraction*100:.1f}%)...")

            def proc_img(p, l):
                nonlocal proc_cnt
                try:
                    img=Image.open(p).convert('RGB');
                    # Resize only if necessary
                    if img.size != (img_size, img_size):
                        img=img.resize((img_size,img_size), Image.Resampling.LANCZOS)
                    img_np=np.array(img);
                    # Basic check for correct shape after potential resize
                    if img_np.shape==(img_size,img_size,3):
                        images.append(img_np); labels.append(l); proc_cnt+=1; return True
                    else:
                        logging.warning(f"Skipping {os.path.basename(p)} due to unexpected shape after processing: {img_np.shape}"); return False
                except Exception as e:
                    logging.error(f"Error processing image {os.path.basename(p)}: {e}"); return False

            all_p=[(p,0) for p in auth_p]+[(p,1) for p in tamp_p]; random.seed(seed); random.shuffle(all_p)
            with tqdm(total=target_f, desc=f"Processing {img_size}x{img_size}") as pbar:
                processed_paths = set() # Avoid duplicates if any
                for p,l in all_p:
                    if p in processed_paths: continue
                    if proc_cnt>=target_f: break
                    if proc_img(p,l):
                        pbar.update(1)
                        processed_paths.add(p)

            if not images: raise RuntimeError("No images were successfully processed!")
            images,labels=np.array(images,dtype=np.uint8),np.array(labels)
            logging.info(f"Successfully processed {len(images)} images. Label distribution: {np.bincount(labels)}")

            # Stratified Split Logic (improved safety)
            lbl_counts=np.bincount(labels); min_samp=min(lbl_counts) if len(lbl_counts)>1 else 0; can_strat=min_samp>=2 # Need at least 2 samples per class
            strat_logic = labels if can_strat else None;
            if not can_strat: logging.warning("Cannot stratify split due to insufficient samples in a class. Using non-stratified split.")

            try:
                # Split Test set first
                tv_img, test_images, tv_lbl, test_labels = train_test_split(images, labels, test_size=test_split, stratify=strat_logic, random_state=seed)

                # Calculate relative validation split size for the remaining data
                rel_val_split = val_split/(train_split+val_split) if (train_split+val_split)>0 else 0

                # Check stratification possibility for train/val split
                tv_lbl_counts=np.bincount(tv_lbl); min_samp_tv=min(tv_lbl_counts) if len(tv_lbl_counts)>1 else 0; can_strat_tv=min_samp_tv>=2 and can_strat # Inherit can_strat
                strat_logic_tv=tv_lbl if can_strat_tv else None;
                if not can_strat_tv and can_strat: logging.warning("Cannot stratify train/validation split. Using non-stratified split for train/val.")

                # Split Train and Validation sets
                train_images, val_images, train_labels, val_labels = train_test_split(tv_img, tv_lbl, test_size=rel_val_split, stratify=strat_logic_tv, random_state=seed)
            except ValueError as e:
                 logging.error(f"Stratified split failed: {e}. Falling back to non-stratified split for all sets.");
                 tv_img, test_images, tv_lbl, test_labels = train_test_split(images, labels, test_size=test_split, random_state=seed)
                 rel_val_split = val_split/(train_split+val_split) if (train_split+val_split)>0 else 0;
                 train_images, val_images, train_labels, val_labels = train_test_split(tv_img, tv_lbl, test_size=rel_val_split, random_state=seed)

            # Save the processed data
            os.makedirs(preprocessed_dir, exist_ok=True);
            logging.info(f"Saving preprocessed data ({img_size}x{img_size}) to {preprocessed_dir}...")
            np.savez_compressed(train_file,images=train_images,labels=train_labels);
            np.savez_compressed(val_file,images=val_images,labels=val_labels);
            np.savez_compressed(test_file,images=test_images,labels=test_labels);
            logging.info("Saved preprocessed data.")
    except Exception as e:
        logging.error(f"Data loading or preprocessing failed: {e}");
        raise e

    logging.info(f"Final Dataset Sizes: Train={len(train_images)}, Validation={len(val_images)}, Test={len(test_images)}")
    logging.info(f"Label Distributions: Train={np.bincount(train_labels)}, Validation={np.bincount(val_labels)}, Test={np.bincount(test_labels)}")
    return train_images, train_labels, val_images, val_labels, test_images, test_labels


# --- Data Augmentations (Unchanged) ---
train_transforms = transforms.Compose([
    # Stronger augmentations can sometimes help
    transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.7, 1.0)), # Slightly wider scale
    transforms.RandomRotation(degrees=20), # More rotation
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1), # More jitter
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3), # Increased vertical flip
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.85, 1.15), shear=15), # More affine
    transforms.ToTensor(),
    transforms.Normalize(mean=config.MEAN, std=config.STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.25), ratio=(0.3, 3.3)) # Increased erasing
])

val_transforms = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.MEAN, std=config.STD)
])

# --- Model Definition (EfficientNet-B0) ---
class EfficientNetB0Model(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        self.model = timm.create_model('efficientnet_b0', pretrained=pretrained)
        # Get the number of input features for the original classifier
        num_ftrs = self.model.classifier.in_features
        # Replace the classifier with a new one matching num_classes
        self.model.classifier = nn.Linear(num_ftrs, num_classes)
        # Optional: Initialize the new classifier layer
        nn.init.xavier_uniform_(self.model.classifier.weight)
        if self.model.classifier.bias is not None:
            nn.init.zeros_(self.model.classifier.bias)

    def forward(self, x):
        # The forward pass uses the underlying timm model
        return self.model(x) # Returns logits directly

# --- Utility Functions (Training, Validation, Evaluation, Plotting - adapted) ---

scaler=GradScaler(enabled=(DEVICE_TYPE=="cuda")) # Enable scaler only if cuda is available

def train_one_epoch(model, dataloader, criterion, optimizer, device, clip_value, scaler):
    model.train(); run_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad() # Initialize gradients once

    for inputs, labels in tqdm(dataloader, "Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)

        # Use autocast for mixed precision
        with torch.amp.autocast(device_type='cuda', enabled=(DEVICE_TYPE == "cuda")):

    # ... forward pass ...
            try:
                outputs = model(inputs) # Model returns only logits now
                loss = criterion(outputs, labels)
            except Exception as e:
                logging.error(f"Error during forward/loss calculation in training: {e}")
                # Optional: skip this batch or handle differently
                continue # Skip to next batch

        # Scale loss and backward pass
        scaler.scale(loss).backward()

        # Unscale gradients before clipping
        scaler.unscale_(optimizer)

        # Clip gradients
        nn.utils.clip_grad_norm_(model.parameters(), clip_value)

        # Optimizer step
        scaler.step(optimizer)

        # Update scaler for next iteration
        scaler.update()

        # Zero gradients for the next batch
        optimizer.zero_grad()

        # --- Statistics ---
        run_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs.detach(), 1) # Use detach() for calculating accuracy
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    if total == 0:
        logging.warning("Training dataloader was empty or all batches failed.")
        return 0.0, 0.0
    epoch_loss = run_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    model.eval(); run_loss, correct, total = 0.0, 0, 0
    with torch.no_grad(): # No gradients needed for validation
        for inputs, labels in tqdm(dataloader, "Validation", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            # Use autocast even in validation for consistency (though grads aren't computed)
            with autocast(device_type=DEVICE_TYPE, enabled=(DEVICE_TYPE=="cuda")):
                try:
                    outputs = model(inputs) # Model returns only logits
                    loss = criterion(outputs, labels)
                except Exception as e:
                    logging.error(f"Error during forward/loss calculation in validation: {e}")
                    continue # Skip to next batch

            # --- Statistics ---
            run_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1) # No need for detach() in torch.no_grad()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        logging.warning("Validation dataloader was empty or all batches failed.")
        return 0.0, 0.0
    epoch_loss = run_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device, patience, clip_value, scaler):
    best_val_acc = 0.0; pat_cnt = 0;
    tr_ls, vl_ls, tr_ac, vl_ac = [], [], [], []
    # Updated model name to reflect single model/phase
    m_name = f"best_model_{config.MODEL_NAME}.pth"

    for epoch in range(num_epochs):
        current_lr = optimizer.param_groups[0]['lr']
        logging.info(f"--- Epoch {epoch+1}/{num_epochs}, LR: {current_lr:.1e} ---")

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, clip_value, scaler)
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        logging.info(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
        print(f"Epoch {epoch+1}/{num_epochs}, LR: {current_lr:.1e}, Train L: {train_loss:.4f}, Acc: {train_acc:.4f} | Val L: {val_loss:.4f}, Acc: {val_acc:.4f}")

        tr_ls.append(train_loss); vl_ls.append(val_loss)
        tr_ac.append(train_acc); vl_ac.append(val_acc)

        # Step the scheduler after validation
        scheduler.step()

        # Early stopping and model saving logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            pat_cnt = 0
            torch.save(model.state_dict(), m_name)
            logging.info(f"Validation accuracy improved to {best_val_acc:.4f}. Saved model to {m_name}")
        else:
            pat_cnt += 1
            logging.info(f"No improvement in validation accuracy. Patience: {pat_cnt}/{patience}")
            if pat_cnt >= patience:
                logging.info("Early stopping triggered.")
                break

    logging.info(f"Training finished. Best Validation Accuracy: {best_val_acc:.4f}")

    # Load the best model found during training for final evaluation
    if os.path.exists(m_name):
        logging.info(f"Loading best model from {m_name}...")
        try:
            # Ensure model is loaded to the correct device
            map_location = torch.device('cpu') if str(device) == 'cpu' else None
            model.load_state_dict(torch.load(m_name, map_location=map_location))
            model.to(device) # Move model back to target device if needed
            logging.info("Best model loaded successfully.")
        except Exception as e:
            logging.error(f"Failed to load best model {m_name}: {e}. Using the current model state.")
    else:
        logging.warning(f"Best model file {m_name} not found. Using the current model state.")

    model.eval() # Ensure model is in eval mode after loading/training
    return model, tr_ls, vl_ls, tr_ac, vl_ac

def evaluate_model(model, test_loader, device, num_classes):
    model.eval(); all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(test_loader, "Testing", leave=False):
            images, labels = images.to(device), labels.to(device)

            with autocast(device_type=DEVICE_TYPE, enabled=(DEVICE_TYPE=="cuda")):
                try:
                    outputs = model(images) # Model returns only logits
                except Exception as e:
                    logging.error(f"Error during forward pass in evaluation: {e}")
                    continue # Skip batch on error

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    if not all_labels:
        logging.error("Evaluation failed: No samples were processed in the test set.")
        return

    logging.info("\n--- Test Set Evaluation Results ---")
    target_names = ['Authentic', 'Tampered'] # Assuming class 0=Auth, 1=Tamp

    try:
        report = classification_report(all_labels, all_preds, target_names=target_names, digits=4, zero_division=0)
        print("Classification Report:")
        print(report)
        logging.info("Classification Report:\n"+report)
    except Exception as e:
        logging.error(f"Could not generate classification report: {e}. Labels present: {np.unique(all_labels)}, Preds present: {np.unique(all_preds)}")

    try:
        cm = confusion_matrix(all_labels, all_preds, normalize='true') # Normalized CM
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt=".2%", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.title(f"Test Confusion Matrix (%) - {config.MODEL_NAME}")
        cm_filename = f'test_confusion_matrix_{config.MODEL_NAME}.png'
        plt.savefig(cm_filename)
        logging.info(f"Saved confusion matrix plot to {cm_filename}")
        plt.show()
    except Exception as e:
        logging.error(f"Could not generate or save confusion matrix: {e}")

# Adjusted plotting functions (removed phase argument)
def plot_loss_curves(train_losses, val_losses, model_name):
    plt.figure(figsize=(10, 6))
    epochs_range = range(1, len(train_losses) + 1)
    plt.plot(epochs_range, train_losses, label='Train Loss', marker='o', linestyle='-')
    plt.plot(epochs_range, val_losses, label='Validation Loss', marker='x', linestyle='--')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Loss Curves - {model_name}')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fname = f'loss_curves_{model_name}.png'
    plt.savefig(fname)
    logging.info(f"Saved loss plot to {fname}")
    plt.show()

def plot_accuracy_curves(train_accs, val_accs, model_name):
    plt.figure(figsize=(10, 6))
    epochs_range = range(1, len(train_accs) + 1)
    plt.plot(epochs_range, train_accs, label='Train Accuracy', marker='o', linestyle='-')
    plt.plot(epochs_range, val_accs, label='Validation Accuracy', marker='x', linestyle='--')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.ylim(0, 1.05) # Ensure y-axis covers 0-1 range
    plt.title(f'Accuracy Curves - {model_name}')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    fname = f'accuracy_curves_{model_name}.png'
    plt.savefig(fname)
    logging.info(f"Saved accuracy plot to {fname}")
    plt.show()


# --- Main Execution Block ---
if __name__ == "__main__":

    # --- Data Loading ---
    logging.info("--- Starting Data Loading ---")
    try:
        train_images, train_labels, val_images, val_labels, test_images, test_labels = load_or_preprocess_data(
            data_path=config.DATA_PATH, preprocessed_dir=config.PREPROCESSED_DIR, img_size=config.IMG_SIZE,
            dataset_fraction=config.DATASET_FRACTION, train_split=config.TRAIN_SIZE, val_split=config.VAL_SIZE,
            test_split=config.TEST_SIZE, seed=config.SEED
        )
    except Exception as e:
        logging.error(f"Data loading failed: {e}. Exiting.")
        exit()

    # Create Datasets
    train_dataset = CASIADataset(train_images, train_labels, train_transforms)
    val_dataset = CASIADataset(val_images, val_labels, val_transforms)
    test_dataset = CASIADataset(test_images, test_labels, val_transforms)

    # --- Weighted Loss Calculation ---
    logging.info("Calculating class weights for weighted loss...")
    train_label_counts = np.bincount(train_labels)
    class_weights = None # Default to None
    if len(train_label_counts) != config.NUM_CLASSES:
        logging.warning(f"Class count mismatch in training data: Found {len(train_label_counts)} classes, expected {config.NUM_CLASSES}. Weighted loss disabled.")
    elif 0 in train_label_counts:
         logging.warning("Found a class with zero samples in the training data. Weighted loss disabled.")
    else:
        # Calculate weights inversely proportional to class frequency
        weight_per_class = len(train_labels) / (config.NUM_CLASSES * train_label_counts)
        class_weights = torch.tensor(weight_per_class, dtype=torch.float).to(DEVICE)
        logging.info(f"Using class weights for loss: {class_weights.cpu().numpy()}")

    # --- Weighted Sampler Calculation ---
    logging.info("Calculating sample weights for weighted sampling...")
    sampler = None; use_sampler = False # Defaults
    if class_weights is not None and len(class_weights) == config.NUM_CLASSES:
        try:
            # Assign the calculated class weight to each sample
            sample_weights_np = class_weights[train_labels].cpu().numpy()
            sample_weights = torch.from_numpy(sample_weights_np).double()
            sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
            use_sampler = True
            logging.info("WeightedRandomSampler created for training data.")
        except IndexError:
             logging.warning("IndexError during sampler weight assignment (likely labels are out of bounds [0, NUM_CLASSES-1]). Weighted sampler disabled.")
        except Exception as e:
             logging.warning(f"Error creating WeightedRandomSampler: {e}. Weighted sampler disabled.")
    else:
         logging.warning("Class weights not available or incorrect count. Weighted sampler disabled.")

    # Create DataLoaders
    num_workers = 2 # Adjust based on your system capabilities (0 is safer for debugging)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        sampler=sampler if use_sampler else None, # Use sampler if created
        shuffle=not use_sampler, # Shuffle only if sampler is NOT used
        pin_memory=True,
        num_workers=num_workers,
        drop_last=True # Drop last incomplete batch if using sampler? Consider this.
    )
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=num_workers)
    logging.info(f"DataLoaders created (Weighted Sampler Active: {use_sampler}, Num Workers: {num_workers}).")

    # --- Model Initialization ---
    logging.info("--- Initializing EfficientNet-B0 Model ---")
    model = EfficientNetB0Model(num_classes=config.NUM_CLASSES, pretrained=True).to(DEVICE)
    logging.info("Model initialized and moved to device.")

    # Loss Function with Weights (if calculated)
    criterion = nn.CrossEntropyLoss(weight=class_weights) # Pass weights (will be None if not calculated)
    logging.info(f"Criterion: CrossEntropyLoss (Weighted: {class_weights is not None})")

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    logging.info(f"Optimizer: AdamW (LR={config.LEARNING_RATE}, WD={config.WEIGHT_DECAY})")

    # Scheduler with Warmup
    # Ensure warmup epochs is not more than total epochs - 1
    actual_warmup_epochs = min(config.WARMUP_EPOCHS, config.EPOCHS - 1)
    if actual_warmup_epochs < 0: actual_warmup_epochs = 0 # Handle EPOCHS=1 case

    scheduler_warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=max(1, actual_warmup_epochs))
    scheduler_cosine = CosineAnnealingLR(optimizer, T_max=max(1, config.EPOCHS - actual_warmup_epochs), eta_min=1e-7) # Small eta_min
    scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[actual_warmup_epochs])
    logging.info(f"Scheduler: SequentialLR (LinearWarmup {actual_warmup_epochs} epochs -> CosineAnnealing)")


    # --- Training ---
    logging.info(f"--- Starting Training for {config.EPOCHS} Epochs ---")
    model, train_losses, val_losses, train_accs, val_accs = train_model(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        config.EPOCHS, DEVICE, config.EARLY_STOPPING_PATIENCE, config.CLIP_GRAD_NORM, scaler
    )

    # --- Plotting Training Curves ---
    if train_losses: # Check if training actually ran
        logging.info("--- Plotting Training Curves ---")
        plot_loss_curves(train_losses, val_losses, config.MODEL_NAME)
        plot_accuracy_curves(train_accs, val_accs, config.MODEL_NAME)
    else:
        logging.warning("Skipping plotting curves as no training history was recorded.")

    # --- Final Evaluation ---
    logging.info("\n--- Final Evaluation on Test Set ---")
    # The best model is already loaded by train_model
    evaluate_model(model, test_loader, DEVICE, config.NUM_CLASSES)

    # --- Attention Plotting Section Removed ---

    logging.info("\n--- EfficientNet-B0 Script Finished ---")

PyTorch Version: 2.6.0+cu124
CUDA Available: False
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<ipython-input-7-9e1f4e3f3bce>:238: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=GradScaler(enabled=(DEVICE_TYPE=="cuda")) # Enable scaler only if cuda is available
Validation:   0%|          | 0/55 [00:00<?, ?it/s]<ipython-input-7-9e1f4e3f3bce>:297: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(device_type=DEVICE_TYPE, enabled=(DEVICE_TYPE=="cuda")):


TypeError: autocast.__init__() got an unexpected keyword argument 'device_type'

In [ ]:
import torch
print(torch.__version__)

2.6.0+cu124
